# Implementing random forests and decision trees from scratch
- First, we need a tree class. Then we write a BuildTree algorithm, which given data, outputs a decision tree corresponding to it.
- Then, include a predictor function, which (given an input) traverses the decision tree to predict an output.
- Turn to random forests : partition data into manageable chunks, train multiple trees, and crowd-source decisions.

In [64]:
import numpy as np
import random as rnd


class Node:
    def __init__(self, value):
        self.value = value  # tuple: [split_feature, split_value, prediction]
        self.left = None
        self.right = None

# -------------------------------------------------------------------------------------------------

class Tree:
    def __init__(self):
        self.root = None  # Starts empty
    def insert(self, value):
        if self.root is None:
            self.root = Node(value)
        else:
            raise ValueError("Node already exists, can only insert at 'root'")
    def predict(self, x):
        def traverse_nodes(active_node, inpu) :
            if active_node.value[0] is None : # meaning split feature is None, i.e, leaf node.
                return active_node.value[2] # POSSIBLY BAD. CHECK IF ROBUST.
            else :
                sp_ft = active_node.value[0]
                sp_val = active_node.value[1]
                if inpu[sp_ft] <= sp_val:
                    return traverse_nodes(active_node.left, inpu)
                else :
                    return traverse_nodes(active_node.right, inpu)
        # ------
        if self.root is None :
            raise ValueError("Node empty")
        else :
            return traverse_nodes(self.root, x)

Now that Tree structure is well-defined, we will define some helper functions. We know, for instance, that when we are at a leaf node (or even an internal one), we can make a prediction for the class of training data that falls under it. The best such prediction turns out to be the mean.
Further, when we will split at a node during training, we will need to know the error derived from branching at a particular split value for a particular feature.

In [65]:
def mean_prediction(y: np.ndarray):
    return np.mean(y)

# -------------------------------------------------------------------------------------------------

def branch_error(y: np.ndarray) -> float:
    if y.shape[0] == 0: # corresponds to a shape like () instead of (num_of_samples,).
        return 0.0
    mean = mean_prediction(y)
    return np.sum((y - mean) ** 2)

# -------------------------------------------------------------------------------------------------

In [66]:

def make_error_array(x: np.ndarray, y: np.ndarray):
    """
    x shape: (num_features, num_samples);
    y shape: (num_samples,).

    Returns:
        split_feature (int)    : index of the best feature to split on
        split_value   (float)  : the threshold value to split at
        left_x, left_y         : samples where x[split_feature] <= split_value
        right_x, right_y       : samples where x[split_feature] >  split_value
    """
    num_features, num_samples = x.shape

    # For each feature, store its possible breakpoints. (find all, remove duplicates, sort).
    # feature_breaks[f]       -> 1D array of breakpoints for feature f
    # feature_break_errors[f] -> 1D array of total errors for each breakpoint of feature f
    feature_breaks = []
    feature_break_errors = []

    for f in range(num_features): # go over all features
        feature_vals = x[f] # shape: (num_samples,), it is the 1-d array of values it takes
        interim_vals  = np.unique(feature_vals) # can contain duplicates, so remove them.
        sorted_vals = np.sort(interim_vals) # no duplicates, may be singleton.


        if len(sorted_vals) == 1 : # if there is only one value taken by y
            breaks = sorted_vals # then break point is just that value
            # At this split value, error will only accumulate for one side.
            errors = np.array([branch_error(y)])

            # ------------------------------------------

            feature_breaks.append(breaks) # remember the break value.
            feature_break_errors.append(errors) # remember the corresponding error.
        else:
            breaks = 0.5 * (sorted_vals[:-1] + sorted_vals[1:])  # shape: (num_samples-1,)
            # this is a clever technique to find breakpoints as the averages of consequent entries, very cool, credit to LLM.

            errors = np.empty(len(breaks)) # prepare to store one error value for each breakpoint
            for b_idx, bp in enumerate(breaks): # fancy syntax (LLM), just splits the data based on its position (<= or >) relative to a splitvalue/breakvalue
                mask        = feature_vals <= bp
                left_error  = branch_error(y[mask])
                right_error = branch_error(y[~mask])
                errors[b_idx] = left_error + right_error

            # ------------------------------------------

            feature_breaks.append(breaks)
            feature_break_errors.append(errors)
    # ---------------------------------------------------------------------------------------------


    # Now, we find the (feature, breakpoint) pair with the lowest total error
    best_error = np.inf
    split_feature = 0
    split_b_idx = 0
    # This is a valid initialization. It can be updated as per below, and if it turns out to be the best split, then even better.
    # It may be a better idea to pick the default randomly, not sure.

    for f in range(num_features): # go over all features,
        local_min_idx = np.argmin(feature_break_errors[f]) # in each feature, locate the best breakpoint by index...
        local_min     = feature_break_errors[f][local_min_idx] # ...and by value.
        if local_min < best_error: # if it is better than the current minimum error, update. This will happen at least once.
            best_error    = local_min
            split_feature = f
            split_b_idx   = local_min_idx
        else: # if it's not better, then leave it be. We know there exists some break with a finite error, so we can always split safely.
            pass

    split_value = feature_breaks[split_feature][split_b_idx]
    # Partition x and y on the best split
    mask        = x[split_feature] <= split_value
    left_x,  left_y  = x[:, mask],  y[mask]
    right_x, right_y = x[:, ~mask], y[~mask]
    return split_feature, split_value, left_x, left_y, right_x, right_y # some of these may be empty.


It may happen that the "best" split gives us a leaf of sorts. What if that proposed leaf is too large? we must account for that to prevent infinite loops.

In [67]:
def build_tree(x: np.ndarray, y: np.ndarray, smallest_class: int = 1) -> Tree:
    """
    Recursively builds a decision tree.

    Each internal node stores:  [split_feature, split_value, None]
    Each leaf node stores:      [None,          None,       right_prediction]

    x shape: (num_features, num_samples)
    y shape: (num_samples,wtf)
    """
    DT = Tree()

    if y.shape == () : # empty tree if no training.
        return DT
    # Stopping condition: too few samples --> leaf node
    if x.shape[1] <= smallest_class:
        prediction = mean_prediction(y)
        # Leaf: split fields unused; both predictions hold the same constant estimate
        DT.insert(np.array([None, None, prediction]))
        return DT

    # Find the best split
    split_feature, split_value, left_x, left_y, right_x, right_y = make_error_array(x, y)
    # We extract the appropriate information using the function we defined above.

    # print(f"sf, sv, lx, rx, = {split_feature}, {split_value}, {left_x}, {right_x}") # sanity check

    if left_x.shape[1] == 0 or right_x.shape[1] == 0 : # should it just be "empty"?
        print(f"Optimal split is a trivial one. Will make a leaf")
        """
        Proposal : If I run the optimization again, I'll get the same result. Better to split arbitrarily here itself.
        Better than to optimize for second best, because second best could have the same issue, and provide no split. Better to move in a suboptimal direction than to not move at all.
        Even then, it may be that all possible splits are trivial. Am leaving this as an oversized leaf node.
        """
        # Select a feature randomly. Select best split value.

        prediction = mean_prediction(y)
        DT.insert(np.array([None, None, prediction]))
        return DT
    else :
        # Recurse into left and right subtrees
        left_subtree  = build_tree(left_x,  left_y,  smallest_class)
        right_subtree = build_tree(right_x, right_y, smallest_class)
        DT.insert(np.array([split_feature, split_value, None]))
        DT.root.left  = left_subtree.root
        DT.root.right = right_subtree.root
    return DT

Now we will turn to random forests.
- Need a sampler to distribute syllabi for each tree to learn.
- Train a specified number of trees, store them in a list. This is the random forest.
- Write a predictor function, which asks each tree, notes the answers down, averages, and reports the answer.

In [68]:
def sampler(num_of_features : int, syllabus_size : int, num_of_trees : int) :
    box = []
    for iterator in range(num_of_trees):
        box.append(rnd.sample(range(num_of_features), syllabus_size))
    return box # This is a list of lists

In [69]:

def train_random_forest(x,y, syllabus_size, num_of_trees, smallest_class = 1)->list[Tree] :
    """
    x.shape = (features,samples)
    y.shape = (samples,1)
    """
    num_of_features = x.shape[0]
    # print(f"num_of_features = {num_of_features}")
    feature_indices_for_each_tree = sampler(num_of_features, syllabus_size, num_of_trees)
    # print(f"sampler gives \n{feature_indices_for_each_tree}\n\n")
    iterator = 0
    my_trees = []
    while iterator < num_of_trees :
        # print(f"x is {x}")
        # print(f"y is {y}")
        # print(f"feature_indices_for_each_tree[iterator] = {feature_indices_for_each_tree[iterator]}")

        temp_x = np.array([x[i] for i in feature_indices_for_each_tree[iterator]])
        temp_y = y # adjustment will occur later, at the level of the number of samples to be used.
        # print(f"temp_x shape is {temp_x.shape}")
        # print()
        # print(f"temp_y shape is {temp_y.shape}")
        # print()
        temp_tree = build_tree(temp_x,temp_y, smallest_class)
        my_trees.append(temp_tree)
        iterator += 1
        # print("Added a tree")
        # print()
        temp_x = None
        temp_y = None
        temp_tree = None
    # Now we have a list of trees. A random forest, if you will
    return my_trees

def predictor_function(ll : list[Tree]) :
    def predictor(x):
        interim_list = [ll[i].predict(x) for i in range(len(ll))] # use the `predict` method for each tree
        return sum(interim_list)/len(interim_list)
    return predictor


Done. Now we can test this structure. For starters, testing on the army classification question from this assignment :

In [86]:
from numpy import array as arrayy
x = arrayy([[175,158, 188, 176, 182, 176, 166, 192], [65, 62, 102, 98, 87, 52, 56, 104]])
y = arrayy([1,0,0,0,1,0,1,1])


"""
CHANGING SYLLABUS SIZE, NUMBER OF TREES AND SMALLEST CLASS WILL AFFECT PERFORMANCE.
"""

mytrees = train_random_forest(x,y, syllabus_size = 1, num_of_trees = 25, smallest_class = 2)


rfpredictor = predictor_function(mytrees)

A = [(arrayy([x[0,i], x[1,i]]) ,y[i]) for i in range(len(y))]

def answer(x : float) : # somewhat arbitrary way to predict 0/1, can work with raw probability also.
    return int(2*x - 0.000001)
print("Probability", " -> " "Predicted Label", " : ", "True label (on training data)")
for i in A :
    print(rfpredictor(i[0]), "         ->         ", answer(rfpredictor(i[0])), "      :      ", i[1])

# ---------------------------------------------------------------------------------------------------------
print("--"*30)



Probability  -> Predicted Label  :  True label (on training data)
1.0          ->          1       :       1
0.44          ->          0       :       0
0.72          ->          1       :       0
0.44          ->          0       :       0
0.72          ->          1       :       1
0.44          ->          0       :       0
1.0          ->          1       :       1
1.0          ->          1       :       1
------------------------------------------------------------


We can compare the individual performance of the trees in our forest also. One can see the predicted values and the true labels in the following :

In [89]:
for tree in mytrees:
    for i in A :
        print(tree.predict(i[0]), "     ->     ", answer(tree.predict(i[0])), "   :   ", i[1])
    print()
    print()

1.0      ->      1    :    1
1.0      ->      1    :    0
1.0      ->      1    :    0
1.0      ->      1    :    0
1.0      ->      1    :    1
1.0      ->      1    :    0
1.0      ->      1    :    1
1.0      ->      1    :    1


1.0      ->      1    :    1
1.0      ->      1    :    0
1.0      ->      1    :    0
1.0      ->      1    :    0
1.0      ->      1    :    1
1.0      ->      1    :    0
1.0      ->      1    :    1
1.0      ->      1    :    1


1.0      ->      1    :    1
0.0      ->      0    :    0
0.5      ->      0    :    0
0.0      ->      0    :    0
0.5      ->      0    :    1
0.0      ->      0    :    0
1.0      ->      1    :    1
1.0      ->      1    :    1


1.0      ->      1    :    1
1.0      ->      1    :    0
1.0      ->      1    :    0
1.0      ->      1    :    0
1.0      ->      1    :    1
1.0      ->      1    :    0
1.0      ->      1    :    1
1.0      ->      1    :    1


1.0      ->      1    :    1
0.0      ->      0    :    0
0.5   